In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import itertools
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
import seaborn as sns
from tqdm import tqdm

from src.analysis import analogy
from src.analysis.state_space import StateSpaceAnalysisSpec, \
    prepare_state_trajectory, flatten_trajectory
from src.datasets.speech_equivalence import SpeechHiddenStateDataset


In [ ]:
base_model = "w2v2_pc_8"

model_class = "discrim-rnn_32-pc-mAP1"
model_name = "word_broad_10frames_fixedlen25"
base_model_class = base_model.rsplit("_", maxsplit=1)[0]


train_dataset = "librispeech-train-clean-100"
# hidden_states_path = f"outputs/hidden_states/{base_model}/{train_dataset}.h5"
hidden_states_path = f"/scratch/jgauthier/{base_model}-{train_dataset}.h5"
state_space_specs_path = f"outputs/analogy_v3/inputs/{train_dataset}/{base_model_class}/state_space_spec.h5"
embeddings_path = f"outputs/model_embeddings/{train_dataset}/{base_model}/{model_class}/{model_name}/{train_dataset}.npy"
# embeddings_path = "ID"
all_cross_instances_path = f"outputs/analogy_v3/inputs/{train_dataset}/{base_model_class}/all_cross_instances.parquet"

output_dir = f"."

pos_counts_path = "data/pos_counts.pkl"

seed = 42

metric = "cosine"

agg_fns = [
    "mean",
]

## Prepare model representations

In [ ]:
if embeddings_path == "ID":
    model_representations = SpeechHiddenStateDataset.from_hdf5(hidden_states_path).states
else:
    with open(embeddings_path, "rb") as f:
        model_representations: np.ndarray = np.load(f)
state_space_spec = StateSpaceAnalysisSpec.from_hdf5(state_space_specs_path)
assert state_space_spec.is_compatible_with(model_representations)

In [ ]:
trajectory_agg = prepare_state_trajectory(model_representations, state_space_spec, 
                                          agg_fn_spec="mean", agg_fn_dimension=1)

In [ ]:
agg, agg_src = flatten_trajectory(trajectory_agg)

## Prepare metadata

In [ ]:
cuts_df = state_space_spec.cuts.xs("phoneme", level="level").drop(columns=["onset_frame_idx", "offset_frame_idx"])
cuts_df["label_idx"] = cuts_df.index.get_level_values("label").map({l: i for i, l in enumerate(state_space_spec.labels)})
cuts_df["frame_idx"] = cuts_df.groupby(["label", "instance_idx"]).cumcount()
cuts_df = cuts_df.reset_index().set_index(["label", "instance_idx", "frame_idx"]).sort_index()

In [ ]:
cut_phonemic_forms = cuts_df.groupby(["label", "instance_idx"]).description.agg(' '.join)

In [ ]:
word_freq_df = pd.read_csv("data/WorldLex_Eng_US.Freq.2.txt", sep="\t", index_col="Word")
# compute weighted average frequency across domains
word_freq_df["BlogFreq_rel"] = word_freq_df.BlogFreq / word_freq_df.BlogFreq.sum()
word_freq_df["TwitterFreq_rel"] = word_freq_df.TwitterFreq / word_freq_df.TwitterFreq.sum()
word_freq_df["NewsFreq_rel"] = word_freq_df.NewsFreq / word_freq_df.NewsFreq.sum()
word_freq_df["Freq"] = word_freq_df[["BlogFreq", "TwitterFreq", "NewsFreq"]].mean(axis=1) \
    * word_freq_df[["BlogFreq", "TwitterFreq", "NewsFreq"]].sum().mean()
word_freq_df["LogFreq"] = np.log10(word_freq_df.Freq)

In [ ]:
all_cross_instances = pd.read_parquet(all_cross_instances_path)

## Behavioral tests

In [ ]:
experiments = {}

In [ ]:
categories = ["comparative", "agentive", "nonmorphemic"]
for c1, c2 in itertools.product(categories, categories):
    exp_name = f"{c1}_to_{c2}"
    experiments[exp_name] = {
        "base_query": f"inflection == '{c1}' and simple == True",
        "inflected_query": f"inflection == '{c2}' and simple == True",
    }
    print(exp_name)

In [ ]:
experiment_results = pd.concat({
    experiment: analogy.run_experiment_equiv_level(
        experiment, config,
        state_space_spec, all_cross_instances,
        agg, agg_src,
        num_samples=1000,
        seed=seed,
        device="cuda")
    for experiment, config in tqdm(experiments.items(), unit="experiment")
}, names=["experiment"])
# experiment_results["correct"] = experiment_results.predicted_label == experiment_results.gt_label
experiment_results

In [ ]:
experiment_results.groupby(["inflection_from", "inflection_to"]).gt_label_rank.agg(["mean", "median"])

In [ ]:
sns.catplot(data=experiment_results, x="inflection_to", hue="inflection_from", y="gt_label_rank", kind="bar")

In [ ]:
sns.catplot(data=experiment_results, x="inflection_to", hue="inflection_from", y="gt_distance", kind="bar")

In [ ]:
sns.catplot(data=experiment_results, x="inflection_to", hue="inflection_from", y="average_precision", kind="bar")

### Save

In [ ]:
experiment_results.to_csv(f"{output_dir}/experiment_results.csv")